# Transformer Ablation Experiments

Notebook này dùng để lấy số liệu điền các TODO liên quan đến **Transformer from scratch** trong báo cáo. Notebook không train hoặc tune trên test. Các thí nghiệm chỉ dùng train/validation:

1. Paper-style Transformer: Post-LN + ReLU, gần sơ đồ gốc trong Attention Is All You Need.
2. Full improved Transformer (optional nếu đã có kết quả thì có thể bỏ qua cell train lại).
3. Ablation: no label smoothing.
4. Ablation: no shared embeddings + no weight tying.

Sau mỗi mô hình, notebook sinh summary trên validation bằng beam search và tính ROUGE.

## 1. Copy project sang /kaggle/working

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

project_src = next(INPUT_ROOT.rglob('summarization_project'))
project_dst = WORK_ROOT / 'summarization_project'

if project_dst.exists():
    shutil.rmtree(project_dst)
shutil.copytree(project_src, project_dst)

print('Copied project from:', project_src)
print('Working project:', project_dst)

In [ ]:
%cd /kaggle/working/summarization_project

## 2. Install dependencies và kiểm tra GPU

In [ ]:
!pip install -q pandas pyarrow sentencepiece rouge-score tqdm pyyaml

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Prepare train/validation data

Cell này chỉ dùng train và validation. Không dùng test trong ablation.

In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
train_file = next(input_root.rglob('train-00000-of-00001.parquet'))
valid_file = next(input_root.rglob('valid-00000-of-00001.parquet'))

print('Train:', train_file)
print('Valid:', valid_file)

In [ ]:
!python scripts/prepare_data.py \
  --train "{train_file}" \
  --valid "{valid_file}" \
  --source-col article \
  --target-col summary

## 4. Train tokenizer và cache train/validation

Tokenizer được train **chỉ từ train.jsonl**. Sau đó cùng tokenizer encode validation.

In [ ]:
!python scripts/train_tokenizer.py \
  --input data/processed/train.jsonl \
  --vocab-size 16000

In [ ]:
!python scripts/tokenize_cache.py \
  --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
  --splits train valid \
  --max-source-len 512 \
  --max-target-len 128

In [ ]:
!cat data/cached/tokenization_stats.json

## 5. Utility: train, generate, evaluate one experiment

In [ ]:
import json
import subprocess
from pathlib import Path

def run_cmd(cmd):
    print('\n' + '=' * 100)
    print(cmd)
    print('=' * 100)
    subprocess.run(cmd, shell=True, check=True)

def train_generate_evaluate(name, extra_train_args=''):
    save_dir = f'checkpoints/{name}'
    pred_path = f'outputs/{name}_valid_predictions_beam.jsonl'
    rouge_path = f'outputs/{name}_valid_rouge_beam.json'

    run_cmd(f'''python scripts/train_baseline.py \
      --epochs 10 \
      --batch-size 4 \
      --d-model 256 \
      --layers 4 \
      --heads 8 \
      --d-ff 1024 \
      --device cuda \
      --save-dir {save_dir} \
      {extra_train_args}''')

    run_cmd(f'''python scripts/generate_summaries.py \
      --checkpoint {save_dir}/best.pt \
      --cache data/cached/valid_tokenized.pkl \
      --processed-jsonl data/processed/valid.jsonl \
      --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
      --method beam \
      --beam-size 4 \
      --length-penalty 0.8 \
      --no-repeat-ngram-size 3 \
      --device cuda \
      --output {pred_path}''')

    run_cmd(f'''python scripts/evaluate_rouge.py \
      --predictions {pred_path} \
      --output {rouge_path}''')

    metrics = json.loads(Path(rouge_path).read_text(encoding='utf-8'))
    history = json.loads(Path(save_dir, 'history.json').read_text(encoding='utf-8'))
    return {
        'name': name,
        'rouge_path': rouge_path,
        'prediction_path': pred_path,
        'final_train_loss': history[-1]['train_loss'],
        'final_val_loss': history[-1]['val_loss'],
        **metrics,
    }

## 6. Train paper-style Transformer

Cấu hình này dùng Post-LN và ReLU để gần với sơ đồ Transformer gốc trong bài báo Attention Is All You Need. Các thành phần training như scheduler và label smoothing vẫn có thể bật/tắt bằng flags ở các cell sau.

In [ ]:
results = []
results.append(train_generate_evaluate('paper_style_transformer', extra_train_args='--norm-type post --activation relu'))

## 7. Optional: train full improved Transformer

Nếu bạn đã có số full model rồi thì có thể bỏ qua cell này. Chạy lại để notebook tự tạo bảng tổng hợp từ đầu.

In [ ]:
results.append(train_generate_evaluate('full_improvements', extra_train_args=''))

## 8. Ablation: no label smoothing

Dùng `--label-smoothing 0.0` để kiểm tra vai trò của label smoothing.

In [ ]:
results.append(train_generate_evaluate('no_label_smoothing', extra_train_args='--label-smoothing 0.0'))

## 9. Ablation: no shared embeddings / no weight tying

Dùng `--no-share-embeddings --no-weight-tying` để kiểm tra tác dụng của giảm tham số và chia sẻ biểu diễn giữa source-target.

In [ ]:
results.append(train_generate_evaluate('no_shared_no_tying', extra_train_args='--no-share-embeddings --no-weight-tying'))

## 10. Summarize ablation results

Bảng này dùng để điền vào `Main results on the validation set` trong báo cáo.

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.DataFrame(results)
cols = ['name', 'examples', 'rouge-1', 'rouge-2', 'rouge-l', 'repetition-3gram', 'compression-ratio', 'final_train_loss', 'final_val_loss']
df = df[cols]
Path('outputs').mkdir(exist_ok=True)
df.to_csv('outputs/transformer_ablation_summary.csv', index=False)
df.to_json('outputs/transformer_ablation_summary.json', orient='records', force_ascii=False, indent=2)
df

In [ ]:
print(df.to_markdown(index=False))

## 11. Count parameters for each variant

Dòng này giúp giải thích vì sao shared embedding và weight tying giảm số tham số.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/summarization_project/src')
from summarization.dataset import SummaryDataset
from summarization.models import TransformerConfig, SummarizationTransformer

ds = SummaryDataset('data/cached/train_tokenized.pkl')

configs = {
    'paper_style_transformer': dict(share_embeddings=True, weight_tying=True, norm_type='post', activation='relu'),
    'full_improvements': dict(share_embeddings=True, weight_tying=True, norm_type='pre', activation='gelu'),
    'no_shared_no_tying': dict(share_embeddings=False, weight_tying=False),
}

for name, kw in configs.items():
    cfg = TransformerConfig(
        vocab_size=ds.vocab_size,
        pad_id=ds.pad_id,
        d_model=256,
        num_encoder_layers=4,
        num_decoder_layers=4,
        num_heads=8,
        d_ff=1024,
        **kw,
    )
    model = SummarizationTransformer(cfg)
    print(name, f'{model.count_parameters():,}')

## 12. Zip outputs để tải về

In [ ]:
%cd /kaggle/working
!zip -r transformer_ablation_results.zip summarization_project/checkpoints summarization_project/outputs
!ls -lh transformer_ablation_results.zip